# Assignment 04: Vector Spatial Analysis

## BIO597 Spatial Analysis of Biodiversity

This assignment asks you to apply spatial joins, buffers, nearest-feature analysis, and polygon overlay to Maine amphibian occurrences.

## Deliverable

Submit a completed notebook containing code, maps, and short interpretations for at least three spatial questions.

## Learning goals

By the end of this assignment, you should be able to:

* connect occurrence points to polygon attributes with a spatial join
* summarize records and species using a new spatial unit
* create and interpret a distance buffer
* calculate distance to the nearest line feature
* use intersection or difference to compare polygon layers

## Data

Use `Amphibians_ME/Amphibians_ME.csv` for occurrence records. The conserved-land, HUC12 watershed, and major-road shapefiles are in `../labs/ME_ShapeFiles`.

Read the `README.md` in that directory before beginning. In particular, remember that the road layer contains primary and secondary roads rather than every local road.

## 1. Import packages

Import `pandas` as `pd` and `geopandas` as `gpd`.

In [1]:
# Import the two packages here.
import pandas as pd
import geopandas as gpd

## 2. Load the occurrence table

Read the tab-delimited amphibian file. Display the first five rows after it loads.

In [2]:
amphibians = pd.read_csv(
    "Amphibians_ME/Amphibians_ME.csv",
    sep="\t",
    low_memory=False,
)

amphibians.head()

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,6520738729,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/39699...,Animalia,Chordata,Amphibia,Anura,Ranidae,Lithobates,Lithobates catesbeianus,...,Chickadee,2026-09-03T19:12:24,CC_BY_NC_4_0,Chickadee,Chickadee,NaN,NaN,2026-09-11T05:24:30.468Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
1,6520735611,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/39672...,Animalia,Chordata,Amphibia,Anura,Ranidae,Aquarana,Aquarana clamitans,...,Noah Nadeau,2026-09-02T18:31:40,CC_BY_NC_4_0,Noah Nadeau,Noah Nadeau,NaN,NaN,2026-09-11T05:38:22.622Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
2,6520729466,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/39753...,Animalia,Chordata,Amphibia,Anura,Ranidae,Boreorana,Boreorana sylvatica,...,Charlotte,2026-09-05T19:23:56,CC_BY_NC_4_0,Charlotte,Charlotte,NaN,NaN,2026-09-11T05:25:44.717Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
3,6520721493,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/39528...,Animalia,Chordata,Amphibia,Anura,Ranidae,Lithobates,Lithobates palustris,...,birdie111,2026-08-28T19:26:37,CC_BY_NC_4_0,birdie111,birdie111,NaN,NaN,2026-09-11T05:24:26.003Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
4,6520718655,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/39691...,Animalia,Chordata,Amphibia,Anura,Ranidae,Lithobates,Lithobates pipiens,...,Wynn Warren,2026-09-03T13:53:41,CC_BY_NC_4_0,Wynn Warren,Wynn Warren,NaN,NaN,2026-09-11T06:08:00.076Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...


## 3. Keep the columns needed for spatial analysis

Keep the record identifier, species, longitude, latitude, coordinate uncertainty, and year. Then display the shape of the smaller table.

In [3]:
columns_to_keep = [
    "gbifID",
    "species",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "year",
]

# Subset amphibians using columns_to_keep.
# Display the number of rows and columns.

amphibians = amphibians[columns_to_keep]
amphibians.head()

,gbifID,species,decimalLongitude,decimalLatitude,coordinateUncertaintyInMeters,year
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,5.0,2026.0
1,6520735611,Aquarana clamitans,-69.649014,44.735713,10.0,2026.0
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,10236.0,2026.0
3,6520721493,Lithobates palustris,-68.636788,44.860466,112.0,2022.0
4,6520718655,Lithobates pipiens,-68.788851,44.990948,1187.0,2026.0


## 4. Prepare the coordinate columns

Convert longitude and latitude to numeric values with `pd.to_numeric()`. Use `errors="coerce"` so unusable values become missing values.

In [4]:
# Convert decimalLongitude to numeric.
# Convert decimalLatitude to numeric.

amphibians["decimalLongitude"] = pd.to_numeric(
    amphibians["decimalLongitude"], errors="coerce"
)
amphibians["decimalLatitude"] = pd.to_numeric(
    amphibians["decimalLatitude"], errors="coerce"
)

## 5. Remove unusable records

Remove rows missing a species name, longitude, or latitude. Use `.copy()` to create an independent table.

In [5]:
amphibians = amphibians.dropna(
    subset=["species", "decimalLongitude", "decimalLatitude"]
).copy()

amphibians.shape

(25586, 6)

## 6. Create an occurrence GeoDataFrame

Use `gpd.points_from_xy()` and assign `EPSG:4326`.

In [6]:
amphibians_gdf = gpd.GeoDataFrame(
    amphibians,
    geometry=gpd.points_from_xy(
        amphibians["decimalLongitude"],
        amphibians["decimalLatitude"],
    ),
    crs="EPSG:4326",
)

amphibians_gdf.head()

,gbifID,species,decimalLongitude,decimalLatitude,coordinateUncertaintyInMeters,year,geometry
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,5.0,2026.0,POINT (-68.66028 44.90255)
1,6520735611,Aquarana clamitans,-69.649014,44.735713,10.0,2026.0,POINT (-69.64901 44.73571)
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,10236.0,2026.0,POINT (-69.88924 44.79832)
3,6520721493,Lithobates palustris,-68.636788,44.860466,112.0,2022.0,POINT (-68.63679 44.86047)
4,6520718655,Lithobates pipiens,-68.788851,44.990948,1187.0,2026.0,POINT (-68.78885 44.99095)


## 7. Load the three supporting layers

Use `gpd.read_file()` to load:

* `Maine_Conserved_Lands.shp`
* `Maine_HUC12_Watersheds.shp`
* `Maine_Primary_Secondary_Roads.shp`

In [7]:
#shape_directory = ""

# Load the conserved-land polygons as conserved.
# Load the watershed polygons as watersheds.
# Load the road lines as roads.

conserved = gpd.read_file("/home/jovyan/BIO597-SpatialBiodiversity/docs/labs/ME_ShapeFiles/Maine_Conserved_Lands.shp")
watersheds = gpd.read_file("/home/jovyan/BIO597-SpatialBiodiversity/docs/labs/ME_ShapeFiles/Maine_HUC12_Watersheds.shp")
roads = gpd.read_file("/home/jovyan/BIO597-SpatialBiodiversity/docs/labs/ME_ShapeFiles/Maine_Primary_Secondary_Roads.shp")

## 8. Inspect the layers

For each GeoDataFrame, display or print its CRS and geometry types. Write one sentence explaining why these details matter.

In [8]:
# Inspect CRS and geometry types here.
print("Occurrences:", amphibians_gdf.geometry.geom_type.unique(), amphibians_gdf.crs)
print("Conserved lands:", conserved.geometry.geom_type.unique(), conserved.crs)
print("Watersheds:", watersheds.geometry.geom_type.unique(), watersheds.crs)
print("Roads:", roads.geometry.geom_type.unique(), roads.crs)

Occurrences: <StringArray>
['Point']
Length: 1, dtype: str EPSG:4326
Conserved lands: <StringArray>
['Polygon', 'MultiPolygon', nan]
Length: 3, dtype: str EPSG:4326
Watersheds: <StringArray>
['Polygon', 'MultiPolygon']
Length: 2, dtype: str EPSG:4326
Roads: <StringArray>
['LineString']
Length: 1, dtype: str EPSG:4269


**Why this matters:** if the crs doesn't match for each layer they cannot be combined. 

## 9. Project every layer

Project all four GeoDataFrames to `EPSG:26919`. Use clear variable names ending in `_projected`.

In [9]:
maine_crs = "EPSG:26919"

amphibians_projected = amphibians_gdf.to_crs(maine_crs)
conserved_projected = conserved.to_crs(maine_crs)
watersheds_projected = watersheds.to_crs(maine_crs)
roads_projected = roads.to_crs(maine_crs)
# Project the occurrences, conserved lands, watersheds, and roads.

# Spatial question 1: Which records occur inside conserved lands?

This section is partly guided. Use a point-in-polygon spatial join with the predicate `within`.

In [10]:
conserved_columns = ["PARCEL_NAM", "CONS1_TYPE", "PUB_ACCESS", "geometry"]

occurrences_in_conserved = gpd.sjoin(
    amphibians_projected,
    conserved_projected[conserved_columns],
    how="inner",
    predicate="within",
)

occurrences_in_conserved.head()

,gbifID,species,decimalLongitude,decimalLatitude,coordinateUncertaintyInMeters,year,geometry,index_right,PARCEL_NAM,CONS1_TYPE,PUB_ACCESS
7,6520700408,Lithobates palustris,-69.865662,44.361917,4.0,2026.0,POINT (431019.46 4912435.936),58,Allen Whitney Memorial Forest,Fee,Contact landowner for additional information
9,6520670325,Boreorana sylvatica,-70.411203,43.454942,4.0,2025.0,POINT (385826.652 4812304.652),8773,NaN,Easement,Contact landowner for additional information
11,6520663387,Lithobates palustris,-69.811887,43.815353,13.0,2026.0,POINT (434705.98 4851685.25),3105,Center Pond Prreserve,Fee,Contact landowner for additional information
12,6520635193,Boreorana sylvatica,-70.389796,43.954013,27411.0,2026.0,POINT (388487.785 4867704.185),5240,NaN,Easement,Restricted - land owner permission required
17,6520610648,Aquarana clamitans,-70.399019,43.934882,NaN,2026.0,POINT (387711.745 4865591.845),10102,L.A. Marshall Co.,Fee,Contact landowner for additional information


## 10. Count unique records and species

Conservation polygons can overlap. First remove repeated `gbifID` values created by overlapping polygon matches. Then calculate:

* the number of unique occurrence records inside conserved lands
* the number of unique species inside conserved lands

In [11]:
# Remove duplicate gbifID values.
unique_conserved_records = occurrences_in_conserved.drop_duplicates(subset="gbifID")



# Calculate the two requested counts.
len(unique_conserved_records)

conserved_species_counts = unique_conserved_records["species"].value_counts()
conserved_species_counts

species
Lithobates catesbeianus       1207
Plethodon cinereus            1110
Anaxyrus americanus           1039
Aquarana clamitans             883
Ambystoma maculatum            630
Boreorana sylvatica            601
Lithobates palustris           575
Pseudacris crucifer            382
Notophthalmus viridescens      332
Dryophytes versicolor          189
Eurycea bislineata             153
Lithobates pipiens             141
Aquarana septentrionalis        76
Hemidactylium scutatum          44
Desmognathus fuscus             36
Ambystoma laterale              32
Gyrinophilus porphyriticus      28
Aquarana catesbeiana            11
Ambystoma jeffersonianum         2
Osteopilus septentrionalis       1
Pseudacris regilla               1
Name: count, dtype: int64

## 11. Map and interpret conserved-land occurrences

Create an interactive map of the matching points colored by species. Include the conserved parcel name in the tooltip.

In [12]:
# Convert the matching points to EPSG:4326 and map them with .explore().
crs_1 = "EPSG:4326"
conserved_projected = unique_conserved_records.to_crs(crs_1)
conserved_projected.explore(
    column="species",
    tooltip=["species", "PARCEL_NAM", "PUB_ACCESS"],
)



**Interpretation:** In two or three sentences, describe the result and identify one limitation of treating an occurrence inside a conserved-land boundary as evidence that conservation caused the occurrence.

Species are widely distributed throughout both conserved and un-conserved lands in Maine; many occurrences are clustered at highly trafficed and protected areas (e.g. Baxter State Park and Acadia NP). This distribution may be caused by sampling bias, where surverys are more frequently done in conserved lands. 

# Spatial question 2: How are occurrences distributed among watersheds?

Choose **three amphibian species** for this analysis. Write their scientific names below and create a GeoDataFrame containing only those species.

**Species 1:** Desmognathus fuscus

**Species 2:** Plethodon cinereus

**Species 3:** Lithobates catesbeianus

In [13]:
selected_species = [
    "Desmognathus fuscus",
    "Plethodon cinereus",
    "Lithobates catesbeianus",
]

# Filter amphibians_projected using .isin(selected_species).
filtered1 = amphibians_projected[amphibians_projected['species'].isin(selected_species)]
filtered2 = amphibians_projected[filtered1 == "TRUE"]
filtered2

selected_gdf = gpd.GeoDataFrame(
    filtered2,
    geometry=gpd.points_from_xy(
        amphibians_projected["decimalLongitude"],
        amphibians_projected["decimalLatitude"],
    ),
    crs= "EPSG:26919",
)



selected_gdf.head()


,gbifID,species,decimalLongitude,decimalLatitude,coordinateUncertaintyInMeters,year,geometry
0,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-68.66 44.903)
1,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-69.649 44.736)
2,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-69.889 44.798)
3,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-68.637 44.86)
4,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-68.789 44.991)


## 12. Assign selected occurrences to HUC12 watersheds

Use `gpd.sjoin()` with the watershed columns `huc12`, `name`, and `geometry`.

In [14]:
# Spatially join your selected occurrence points to watersheds.
watershed_columns = ["huc12", "name", "geometry"]

occurrences_by_watershed = gpd.sjoin(
    selected_gdf,
    watersheds_projected[watershed_columns],
    how="inner",
    predicate="within",
)

occurrences_by_watershed.head()

# Display the first five rows of the result.

,gbifID,species,decimalLongitude,decimalLatitude,coordinateUncertaintyInMeters,year,geometry,index_right,huc12,name


## 13. Summarize by watershed

Create a table with one row per HUC12 watershed. Include:

* the number of unique occurrence records
* the number of unique selected species

Use the Lab 04 workflow as a model, but write this code yourself.

In [15]:
# Group by huc12 and name.
watershed_summary = occurrences_by_watershed.groupby(
    ["huc12", "name"]
).agg(
    records=("gbifID", "nunique"),
    species_count=("species", "nunique"),
)

watershed_summary = watershed_summary.reset_index()
watershed_summary.head()

# Calculate unique records and unique species.
watershed_summary.sort_values("species_count", ascending=False).head(10)

# Reset the index.
watershed_summary = watershed_summary.reset_index()


## 14. Map the watershed result

Merge your summary table back into the watershed polygons. Make a map colored by either record count or species count.

In [16]:
# Merge the summary with watersheds_projected.
watersheds_with_counts = watersheds_projected.merge(
    watershed_summary,
    on=["huc12", "name"],
    how="left",
)

# Convert to EPSG:4326 and map with .explore().
watersheds_with_counts.to_crs("EPSG:4326").explore(
    column="species_count",
    tooltip=["name", "huc12", "records", "species_count"],
)

ValueError: max() iterable argument is empty

**Interpretation:** Which watersheds have the most records or species? Give one ecological explanation and one sampling explanation for the pattern.

# Spatial question 3: Which records are within 500 m of a major road?

Create a 500 m buffer around the projected roads. Combine the overlapping buffers with `union_all()`, then place that geometry in a one-row GeoDataFrame.

In [ ]:
road_buffer_geometry = roads_projected.buffer(500)
road_buffer_union = road_buffer_geometry.union_all()

road_buffer = gpd.GeoDataFrame(
    {"buffer_m": [500]},
    geometry=[road_buffer_union],
    crs=maine_crs,
)

## 15. Find occurrences inside the road buffer

Use a spatial join to find all occurrence points within the combined road buffer.

In [ ]:
# Spatially join amphibians_projected to road_buffer.
occurrences_near_roads = gpd.sjoin(
    amphibians_projected,
    road_buffer,
    how="inner",
    predicate="within",
)

# Display the shape of the result.
occurrences_near_roads.shape


## 16. Summarize the road-buffer result

Report:

* the number of unique species with at least one record within 500 m of a major road
* the number of records within 500 m for each species
* the percentage of all occurrence records that are within 500 m

In [ ]:
# Calculate the three requested summaries.

#num species near road
species_near_roads = occurrences_near_roads["species"].nunique()
species_near_roads

#num records w/i 500m for each sp. 
occurrences_near_roads["species"].value_counts()

#percentage
percentage = (occurrences_near_roads["species"].value_counts())/(amphibians_projected["species"].value_counts())
percentage


**Interpretation:** Does this result demonstrate that amphibians prefer roads? Explain why or why not, and mention how observer access could affect the pattern.

This result cannot causally explain that amphibians prefer habitat near roads, as the surrounding plant community and hydrology of the habitat confounds this. Sampling bias is also certainly influencing these results, as surverys are easily done along roadways in comparison to more remote localities. 

# Spatial question 4: How far are records from the nearest major road?

Use `gpd.sjoin_nearest()` to add the nearest road and its distance to every occurrence. Keep `FULLNAME`, `MTFCC`, and `geometry` from the road layer. Name the distance column `distance_to_road_m`.

In [ ]:
road_columns = ["FULLNAME", "MTFCC", "geometry"]

# Complete the nearest spatial join.
nearest_roads = gpd.sjoin_nearest(
    amphibians_projected,
    roads_projected[road_columns],
    how="left",
    distance_col="distance_to_road_m",
)

nearest_roads.head()

## 17. Compare nearest-road distance among your three species

Filter the nearest-road result to your three selected species. Calculate the median distance to the nearest major road for each species.

In [ ]:
# Filter with .isin(selected_species).
# Group by species and calculate the median distance_to_road_m.

**Interpretation:** Which selected species has records closest to major roads? Give at least one reason this could reflect the observation process instead of species biology.

# Spatial question 5: Conserved area in one watershed

Choose one HUC12 watershed from your summary. Store its HUC12 code below, then create a one-row or small GeoDataFrame containing that watershed.

In [ ]:
chosen_huc12 = "010200010703"

# Filter watersheds_projected to chosen_huc12.
study_watershed = watersheds_projected[
    watersheds_projected["huc12"] == chosen_huc12
].copy()

study_watershed[["huc12", "name"]]

## 18. Intersect conserved lands with the chosen watershed

Use `gpd.overlay()` with `how="intersection"`. Calculate the total intersecting conserved area in square kilometers.

In [ ]:
# Intersect conserved_projected with your chosen watershed.
conserved_projected.to_crs("EPSG:26919")
study_watershed.to_crs("EPSG:26919")

conserved_in_watershed = gpd.overlay(
    conserved_projected,
    study_watershed,
    how="intersection",
)

conserved_in_watershed.shape
# Sum polygon area and convert square meters to square kilometers.

## 19. Find the part outside conserved lands

Dissolve the intersecting conservation polygons, then use `gpd.overlay()` with `how="difference"` to retain the part of the watershed outside those polygons.

In [ ]:
# Dissolve the intersecting conserved polygons.
conserved_union = conserved_in_watershed.dissolve()

unconserved_part = gpd.overlay(
    study_watershed,
    conserved_union,
    how="difference",
)

# Calculate the difference between the watershed and dissolved polygons.
conserved_area_km2 = conserved_in_watershed.area.sum() / 1_000_000
watershed_area_km2 = study_watershed.area.sum() / 1_000_000



## 20. Map and interpret the overlay

Make an interactive map showing either the conserved intersection or the area outside conserved lands. Report the approximate percentage of the watershed inside conserved lands.

In [ ]:
# Calculate the percentage.
100 * conserved_area_km2 / watershed_area_km2

# Convert your chosen result to EPSG:4326 and map it.
m = study_watershed.explore()
unconserved_part.explore(m=m)

**Interpretation:** What does intersection retain? What does difference retain? State one limitation of the conserved-land boundaries.

## Final reflection

Answer each question in two or three sentences.

1. Which spatial operation changed the unit of analysis most clearly?
The creation and overlap of a single watershed and the conserved area most clearly changed the nature of the analysis. The reduction of the spatial scale observed, along with the results being calculated as an area rather than species counts made this analysis different. 


2. Which of your results is most sensitive to the choice of CRS?
All analyses using multiple geospatial data layers are sensitive to any changes to CRS. If there is a mismatch between the CRS for two layers, then the analysis cannot run. 


3. Which result is most likely to be affected by uneven sampling effort?
The distribution of occurrences near roadways would most likely be affected. It is likely that there is a higher degree of sampling effort on more accessible roads. 


4. Propose one additional ecological or conservation question that could be answered with these layers.
Can sampling bias be quantified as a function of an occurrences proximity to roads and conserved areas?

## Submit your work

Run the notebook from top to bottom. Make sure all maps display and all written interpretations are complete. Then commit and push the completed notebook to your class GitHub repository.

```bash
git add docs/assignments/Assignment-04-VectorSpatialAnalysis.ipynb
git commit -m "Complete vector spatial analysis assignment"
git push
```